# Task 2.1 — Dataset Selection and Setup
**Paper:** Agarwal, A., Xie, B., Vovsha, I., Rambow, O., & Passonneau, R. (2011). *Sentiment Analysis of Twitter Data.* ACL Workshop on Language in Social Media (LSM 2011), pp. 30–38.

---

## Global Config
All hyperparameters and seeds in one place.

In [1]:
RANDOM_SEED       = 42    # used in all .sample() calls
SAMPLES_PER_CLASS = 1709  # paper Section 3: 1709 per class

import random, numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f'Seed: {RANDOM_SEED} | Samples per class: {SAMPLES_PER_CLASS}')

Seed: 42 | Samples per class: 1709


## Dependencies

In [2]:
import subprocess, sys

# Install packages
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'nltk', 'scikit-learn', 'numpy', 'matplotlib',
    'seaborn', 'pandas', '--break-system-packages', '-q'])

# Download NLTK corpora
import nltk
for pkg in ['twitter_samples', 'averaged_perceptron_tagger',
            'wordnet', 'stopwords', 'punkt', 'punkt_tab',
            'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)

print('All dependencies ready.')


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


All dependencies ready.


---

## Dataset Justification

**What it is:**
- NLTK `twitter_samples` — 10,000 real tweets (5,000 positive, 5,000 negative)
- Collected from the Twitter Streaming API; bundled with NLTK, no manual download needed
- Labelled via distant supervision: positive emoticons `:)` → positive, negative emoticons `:(` → negative

**Why it is a reasonable testbed:**
- It is actual Twitter data with the same noise the paper's method targets: emoticons, hashtags, @-targets, abbreviations, URLs, repeated characters
- The paper's full 5-step preprocessing pipeline (Section 4) applies directly to this data
- No other widely available dataset is this close to the paper's own corpus in domain and format

**Limitations vs. the paper's dataset:**
- **Binary only** — no neutral class; only the 2-way task (Section 8.1) is reproducible, not the 3-way task
- **Distant supervision labels** — labelled by emoticon heuristic, not by human annotators; may slightly overestimate performance on emoticon-heavy tweets
- **Size mismatch** — 10,000 tweets available; subsampled to 1,709 per class (3,418 total) to match the paper's balanced setup (Section 3)

---

## Step 1 — Load Raw Data

In [3]:
import pandas as pd
from nltk.corpus import twitter_samples

pos_tweets = twitter_samples.strings('positive_tweets.json')  # 5,000 tweets
neg_tweets = twitter_samples.strings('negative_tweets.json')  # 5,000 tweets

# Label: 1 = positive, 0 = negative
df = pd.DataFrame({
    'text':  pos_tweets + neg_tweets,
    'label': [1]*len(pos_tweets) + [0]*len(neg_tweets)
})

print(f'Loaded: {len(df)} tweets')
print(df['label'].value_counts())
print(f'\nSample: {df["text"].iloc[0]}')

Loaded: 10000 tweets
label
1    5000
0    5000
Name: count, dtype: int64

Sample: #FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)


**What this does:**
- Loads 5,000 positive + 5,000 negative tweets from NLTK
- Assigns integer labels: `1` = positive, `0` = negative — matching the 2-way task in Section 8.1

## Step 2 — Balance the Dataset

In [4]:
# Subsample to 1,709 per class — matches paper Section 3
df_pos = df[df['label']==1].sample(n=SAMPLES_PER_CLASS, random_state=RANDOM_SEED)
df_neg = df[df['label']==0].sample(n=SAMPLES_PER_CLASS, random_state=RANDOM_SEED)

# Shuffle to remove ordering bias
df_balanced = (
    pd.concat([df_pos, df_neg])
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

print(f'Balanced size: {len(df_balanced)}')
print(df_balanced['label'].value_counts())

Balanced size: 3418
label
0    1709
1    1709
Name: count, dtype: int64


**What this does:**
- Subsamples to 1,709 per class — identical to the paper's balanced setup (Section 3)
- Sets the chance baseline at exactly 50%, matching Table 5 of the paper
- Shuffles to remove any ordering bias before train/test splitting

## Step 3 — Preprocessing Pipeline

Five steps from **Section 4** of the paper:

| Step | Operation | Paper ref |
|------|-----------|----------|
| 1 | Emoticons → polarity tags (`\|\|P\|\|`, `\|\|N\|\|`, `\|\|ExP\|\|`, ...) | Section 4a, Table 2 |
| 2 | URLs → `\|\|U\|\|` | Section 4b |
| 3 | @targets → `\|\|T\|\|` | Section 4c |
| 4 | Negations (`not`, `never`, `n't`, ...) → `NOT` | Section 4d |
| 5 | Repeated chars collapsed to 3 (`coooool` → `coool`) | Section 4e |

In [5]:
import re

# Emoticon → polarity tag (Section 4a, Table 2)
# Double-backslashes needed: r'' prefix is lost in .ipynb JSON serialisation
EMOTICON_MAP = [
    (':-D|:D|=D',                  '||ExP||'),  # Extremely Positive
    (':-\\)|:\\)|:o\\)|=\\)|8\\)', '||P||'),    # Positive
    ('D:|D;|D=|DX',                '||ExN||'),  # Extremely Negative
    (':-\\(|:\\(',                 '||N||'),    # Negative
    (':\\|',                       '||Neu||'),  # Neutral
]

# Negation words (Section 4d)
NEGATIONS = {'not','no','never',"n't",'cannot',"can't","won't",
             "didn't","doesn't","isn't","wasn't","weren't","hadn't"}

def preprocess_tweet(tweet):
    for pattern, tag in EMOTICON_MAP:          # Step 1 — emoticons first
        tweet = re.sub(pattern, ' '+tag+' ', tweet)
    tweet = re.sub('https?://\\S+|www\\.\\S+', '||U||', tweet)  # Step 2 — URLs
    tweet = re.sub('@\\w+', '||T||', tweet)                      # Step 3 — targets
    tokens = ['NOT' if t.lower() in NEGATIONS else t             # Step 4 — negations
              for t in tweet.split()]
    tweet = re.sub('(.)\\1{3,}', '\\1\\1\\1', ' '.join(tokens)) # Step 5 — repeated chars
    return tweet.strip()

# Sanity check — verify each step works before running on full dataset
tests = [
    ('I love this :)',           'emoticon +ve'),
    ('this is awful :(',         'emoticon -ve'),
    ('check http://t.co/x now',  'URL'),
    ('thanks @john !',           'target'),
    ('I am not happy',           'negation'),
    ('sooooo good',              'repeated chars'),
]
print('--- Sanity check ---')
for raw, label in tests:
    print(f'  [{label:<15}]  {raw!r:<35} -> {preprocess_tweet(raw)!r}')

# Apply to full dataset
df_balanced['processed'] = df_balanced['text'].apply(preprocess_tweet)
print(f'\nDone. {len(df_balanced)} tweets preprocessed.')

--- Sanity check ---
  [emoticon +ve   ]  'I love this :)'                    -> 'I love this ||P||'
  [emoticon -ve   ]  'this is awful :('                  -> 'this is awful ||N||'
  [URL            ]  'check http://t.co/x now'           -> 'check ||U|| now'
  [target         ]  'thanks @john !'                    -> 'thanks ||T|| !'
  [negation       ]  'I am not happy'                    -> 'I am NOT happy'
  [repeated chars ]  'sooooo good'                       -> 'sooo good'

Done. 3418 tweets preprocessed.


**What this does:**
- Implements all 5 preprocessing steps from Section 4 in order
- Emoticons run **first** — before URL/target regex — so `:` in emoticons is not consumed by other patterns
- Repeated chars collapsed to **3** (not 2) — the paper notes this specific choice to distinguish emphasis (`coool`) from normal spelling (Section 4e)
- Sanity check confirms each step works on known examples before touching the full dataset

## Step 4 — Dataset Statistics
Compare token composition to Table 3 in the paper.

In [6]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

# .split() not TweetTokenizer — tokenizer splits '||P||' on '|', making tag counts zero
all_tokens = [t for tweet in df_balanced['processed'] for t in tweet.split()]
n = len(all_tokens)

rows = [
    ('Total tokens',        n),
    ('Stop words',          sum(1 for t in all_tokens if t.lower() in stop_words)),
    ('URLs  ||U||',         all_tokens.count('||U||')),
    ('Targets ||T||',       all_tokens.count('||T||')),
    ('Positive emoticons',  all_tokens.count('||P||') + all_tokens.count('||ExP||')),
    ('Negative emoticons',  all_tokens.count('||N||') + all_tokens.count('||ExN||')),
    ('Negations NOT',       all_tokens.count('NOT')),
    ('Hashtags',            sum(1 for t in all_tokens if t.startswith('#'))),
]

print(f'{"Metric":<24} {"Count":>7}  {"Our %":>7}  {"Paper %":>8}')
print('-' * 52)
paper_vals = {'Stop words':'38.3%', 'URLs  ||U||':'~1%',
              'Targets ||T||':'~2%', 'Negations NOT':'1.2%'}
for label, count in rows:
    pct  = f'{100*count/n:.1f}%' if label != 'Total tokens' else ''
    ref  = paper_vals.get(label, '')
    print(f'{label:<24} {count:>7}  {pct:>7}  {ref:>8}')

print('\nRef: paper Table 3 — 38.3% stop words, ~4.2% Twitter tags, 1.2% negations')

Metric                     Count    Our %   Paper %
----------------------------------------------------
Total tokens               40249                   
Stop words                 13258    32.9%     38.3%
URLs  ||U||                  607     1.5%       ~1%
Targets ||T||               2763     6.9%       ~2%
Positive emoticons          1771     4.4%          
Negative emoticons          1770     4.4%          
Negations NOT                520     1.3%      1.2%
Hashtags                     743     1.8%          

Ref: paper Table 3 — 38.3% stop words, ~4.2% Twitter tags, 1.2% negations


**What this does:**
- Counts key token categories across the preprocessed dataset, mirroring Table 3 of the paper
- Prints our percentages alongside the paper's values so differences are immediately visible
- Uses `.split()` not `TweetTokenizer` — the tokenizer splits `||P||` on `|` characters, making all tag counts incorrectly zero

## Step 5 — Save Processed Dataset

In [7]:
import os
os.makedirs('data', exist_ok=True)

# Single save point — all downstream notebooks load from this file
df_balanced.to_csv('data/tweets_balanced_preprocessed.csv', index=False)
print(f'Saved {len(df_balanced)} tweets -> data/tweets_balanced_preprocessed.csv')
print(df_balanced[['text','processed','label']].head(3))

Saved 3418 tweets -> data/tweets_balanced_preprocessed.csv
                                                text  \
0  Not 12 hours after my sister-in-law installed ...   
1  @DaShawnBullfrog @PapaHogieBear I just got her...   
2  Hi I'm Madison, I'm 13 not supposed to have tw...   

                                           processed  label  
0  NOT 12 hours after my sister-in-law installed ...      0  
1         ||T|| ||T|| I just got here ||N|| come!!!'      0  
2  Hi I'm Madison, I'm 13 NOT supposed to have tw...      1  


**What this does:**
- Saves the balanced, preprocessed dataset as a CSV to `data/`
- All downstream notebooks (task_2_2, task_2_3, task_3_1, task_3_2) load from this single file — ensures every task operates on identical input